# Introduction to Machine Learning — Tutorial

**Primary outcome:** Walk through a complete ML workflow end-to-end — from raw data to a trained model — understanding what happens at each step and why.

**Time:** 60–90 minutes

---

## What you'll learn

1. What machine learning is and its three main types
2. The difference between regression and classification
3. How to move through a full ML pipeline step by step
4. How to preprocess real data, engineer features, and evaluate a model
5. What the bias-variance tradeoff means in practice
6. How to use learning curves to diagnose model problems
7. How to compare two different models

---

**Datasets used:**
- Titanic (`seaborn.load_dataset('titanic')`) — real survival data, 891 passengers
- Iris (`sklearn.datasets.load_iris()`) — classic flower classification dataset
- Synthetic datasets for concept illustration

In [ ]:
# --- Imports and global settings ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Reproducibility
np.random.seed(42)

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries loaded successfully.")

---

## Section 1 — What is Machine Learning?

Machine learning is the practice of building systems that **learn patterns from data** rather than following hand-written rules.

Instead of: `if height > 180 and weight > 90: label = 'rugby player'`  
You let an algorithm find those boundaries automatically from labelled examples.

### The three types

| Type | What it learns from | Goal | Example |
|------|---------------------|------|---------|
| **Supervised** | Labelled input–output pairs | Predict the label for new inputs | Spam detection → classify email as spam/not spam |
| **Unsupervised** | Unlabelled data only | Find structure or groupings | Customer segmentation → cluster shoppers by behaviour |
| **Reinforcement** | Rewards and penalties from an environment | Learn a policy that maximises reward | Game-playing agent → AlphaGo, robotics control |

### More real-world examples

| Problem | Type | Task |
|---------|------|------|
| Predict house price | Supervised | Regression |
| Detect credit card fraud | Supervised | Classification |
| Group news articles by topic | Unsupervised | Clustering |
| Reduce 100 features to 2 for plotting | Unsupervised | Dimensionality reduction |
| Train a robot to walk | Reinforcement | Policy learning |
| Recommend movies | Supervised / Unsupervised | Ranking / Collaborative filtering |

> **This tutorial focuses on supervised learning** — the most common type in industry.

### Illustrative example: KNN does what you'd do by hand

Imagine you have 10 people with their height and weight, and you know which sport each plays. When a new person arrives, you find the **nearest neighbours** and vote. That's exactly what `KNeighborsClassifier` does — no magic, just geometry.

In [ ]:
# 10 labelled examples: [height_cm, weight_kg]
data = [
    [170, 65], [175, 70], [180, 85], [185, 90], [190, 95],   # basketball
    [160, 55], [158, 50], [162, 52], [155, 48], [163, 54],   # gymnastics
]
labels = ['basketball'] * 5 + ['gymnastics'] * 5

X_demo = np.array(data)
y_demo = np.array(labels)

# New person: 172 cm, 68 kg — which sport?
new_person = np.array([[172, 68]])

# --- By hand: sort by Euclidean distance and take majority vote ---
distances = np.sqrt(((X_demo - new_person) ** 2).sum(axis=1))
sorted_idx = np.argsort(distances)

print("=== Manual KNN (k=3) ===")
print(f"{'Rank':<6} {'Height':>8} {'Weight':>8} {'Distance':>10} {'Label':>12}")
print("-" * 48)
for rank, i in enumerate(sorted_idx[:3], 1):
    print(f"{rank:<6} {X_demo[i,0]:>8} {X_demo[i,1]:>8} {distances[i]:>10.2f} {y_demo[i]:>12}")

from collections import Counter
votes = Counter(y_demo[sorted_idx[:3]])
manual_prediction = votes.most_common(1)[0][0]
print(f"\nVotes: {dict(votes)}")
print(f"Manual prediction  → {manual_prediction}")

# --- sklearn does the same thing ---
knn = KNeighborsClassifier(n_neighbors=3)  # k=3: look at 3 nearest neighbours
knn.fit(X_demo, y_demo)
sklearn_prediction = knn.predict(new_person)[0]
print(f"sklearn prediction → {sklearn_prediction}")
print(f"\nMatch: {manual_prediction == sklearn_prediction}")

---

## Section 2 — Supervised Learning: Regression vs Classification

Supervised learning splits into two families depending on what you are predicting:

| | **Regression** | **Classification** |
|-|----------------|--------------------|
| Output | Continuous number | Discrete category |
| Question | *How much?* / *How many?* | *Which class?* |
| Examples | House price, temperature, salary | Spam/not spam, disease diagnosis, digit recognition |
| Common metrics | MSE, RMSE, MAE, R² | Accuracy, Precision, Recall, F1 |

The code below generates one synthetic dataset of each type and plots them side by side so you can see the difference visually.

In [ ]:
np.random.seed(42)

# --- Regression dataset: house size → price ---
n = 80
house_size = np.random.uniform(50, 250, n)                      # m²
house_price = 1500 * house_size + np.random.normal(0, 20000, n) # €

# --- Classification dataset: study hours + sleep hours → pass/fail ---
hours_study = np.random.uniform(0, 10, n)
hours_sleep  = np.random.uniform(4, 10, n)
# Simple rule with noise
pass_score = 0.6 * hours_study + 0.4 * hours_sleep + np.random.normal(0, 0.8, n)
pass_fail = (pass_score > 5).astype(int)  # 1 = pass, 0 = fail

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: regression
axes[0].scatter(house_size, house_price / 1000, alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)
axes[0].set_title('Regression — predict a number\n(house size → price)', fontsize=13)
axes[0].set_xlabel('House size (m²)')
axes[0].set_ylabel('Price (€ thousands)')
axes[0].annotate('Output: continuous\ne.g. €183,400', xy=(0.05, 0.85), xycoords='axes fraction',
                 fontsize=10, color='steelblue',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.4))

# Right: classification
colors = ['#e74c3c' if p == 0 else '#2ecc71' for p in pass_fail]
axes[1].scatter(hours_study, hours_sleep, c=colors, alpha=0.7, edgecolors='white', linewidth=0.5)
axes[1].set_title('Classification — predict a category\n(study + sleep → pass/fail)', fontsize=13)
axes[1].set_xlabel('Hours studied')
axes[1].set_ylabel('Hours slept')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Pass'),
                   Patch(facecolor='#e74c3c', label='Fail')]
axes[1].legend(handles=legend_elements)
axes[1].annotate('Output: category\ne.g. Pass or Fail', xy=(0.05, 0.85), xycoords='axes fraction',
                 fontsize=10, color='#27ae60',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='#d5f5e3', alpha=0.6))

plt.tight_layout()
plt.suptitle('Two types of supervised learning', fontsize=14, y=1.02)
plt.show()

print(f"Regression dataset : {n} samples, output range €{house_price.min()/1000:.0f}k – €{house_price.max()/1000:.0f}k")
print(f"Classification data: {n} samples, {pass_fail.sum()} pass ({pass_fail.mean()*100:.0f}%), {(1-pass_fail).sum()} fail")

---

## Section 3 — The ML Workflow

A real ML project follows a repeating cycle of seven steps. Skipping any step causes problems downstream — garbage in, garbage out.

```
1. Define the problem    → What question are we answering? What does success look like?
2. Collect data          → Gather raw data from databases, APIs, sensors, surveys.
3. Explore the data      → Understand distributions, missing values, outliers (EDA).
4. Preprocess & engineer → Clean, encode, scale, create new features.
5. Train a model         → Choose an algorithm; fit it on training data.
6. Evaluate & iterate    → Measure performance on held-out data; tune hyperparameters.
7. Deploy & monitor      → Serve predictions; watch for data drift over time.
```

> In this tutorial we work through steps 2–6 on the Titanic dataset.

In [ ]:
# Print the workflow as a checklist
steps = [
    ("1", "Define the problem",   "Will we predict survival (yes/no)? Binary classification."),
    ("2", "Collect / load data",  "Load Titanic dataset — 891 passengers, 12 columns."),
    ("3", "Exploratory analysis", "Check shape, nulls, distributions, survival rates."),
    ("4", "Preprocess & engineer","Fill missing values, encode categories, create features."),
    ("5", "Train/test split",     "Hold out 20% so we have an honest evaluation set."),
    ("6", "Scale features",       "Standardise on train only to avoid data leakage."),
    ("7", "Fit & evaluate model", "LogisticRegression → accuracy, confusion matrix, report."),
]

print("=" * 65)
print("  ML WORKFLOW CHECKLIST")
print("=" * 65)
for num, title, desc in steps:
    print(f"  [ ] Step {num}: {title}")
    print(f"        {desc}")
    print()
print("=" * 65)

---

## Section 4 — Steps 1–2: Load Data & EDA

We will use the **Titanic** dataset — passenger manifests from the 1912 sinking. The target is `survived` (1 = survived, 0 = did not).

This is a real-world dataset: it has missing values, mixed data types, and irrelevant columns — all things you will encounter in practice.

In [ ]:
# Load Titanic dataset
titanic = sns.load_dataset('titanic')

print("Shape:", titanic.shape)
print("\nColumns:", list(titanic.columns))
print("\nData types:")
print(titanic.dtypes)
print("\nFirst 5 rows:")
titanic.head()

In [ ]:
# --- Missing value audit ---
print("=== Null counts per column ===")
null_counts = titanic.isnull().sum()
null_pct    = (null_counts / len(titanic) * 100).round(1)
null_df = pd.DataFrame({'missing': null_counts, 'pct': null_pct})
null_df = null_df[null_df['missing'] > 0].sort_values('missing', ascending=False)
print(null_df.to_string())

# --- Basic statistics ---
print("\n=== Survival rate ===")
print(titanic['survived'].value_counts())
print(f"Overall survival rate: {titanic['survived'].mean()*100:.1f}%")

print("\n=== Survival by passenger class ===")
print(titanic.groupby('pclass')['survived'].mean().round(3))

print("\n=== Survival by sex ===")
print(titanic.groupby('sex')['survived'].mean().round(3))

In [ ]:
# 4-panel EDA figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Titanic — Exploratory Data Analysis', fontsize=15)

# Panel 1: Survival counts
titanic['survived'].value_counts().rename({0: 'Did not survive', 1: 'Survived'}).plot(
    kind='bar', ax=axes[0, 0], color=['#e74c3c', '#2ecc71'], edgecolor='white', rot=0
)
axes[0, 0].set_title('Survival count')
axes[0, 0].set_ylabel('Number of passengers')
axes[0, 0].set_xlabel('')
for bar in axes[0, 0].patches:
    axes[0, 0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                    int(bar.get_height()), ha='center', va='bottom', fontsize=11)

# Panel 2: Age distribution by survival
titanic[titanic['survived'] == 0]['age'].dropna().plot(
    kind='hist', bins=25, ax=axes[0, 1], alpha=0.6, color='#e74c3c', label='Did not survive'
)
titanic[titanic['survived'] == 1]['age'].dropna().plot(
    kind='hist', bins=25, ax=axes[0, 1], alpha=0.6, color='#2ecc71', label='Survived'
)
axes[0, 1].set_title('Age distribution by survival')
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend()

# Panel 3: Survival rate by passenger class
survival_by_class = titanic.groupby('pclass')['survived'].mean() * 100
survival_by_class.plot(kind='bar', ax=axes[1, 0], color='steelblue', edgecolor='white', rot=0)
axes[1, 0].set_title('Survival rate by passenger class')
axes[1, 0].set_xlabel('Passenger class')
axes[1, 0].set_ylabel('Survival rate (%)')
axes[1, 0].set_ylim(0, 100)
for bar in axes[1, 0].patches:
    axes[1, 0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=11)

# Panel 4: Survival rate by sex
survival_by_sex = titanic.groupby('sex')['survived'].mean() * 100
survival_by_sex.plot(kind='bar', ax=axes[1, 1], color=['#9b59b6', '#e67e22'], edgecolor='white', rot=0)
axes[1, 1].set_title('Survival rate by sex')
axes[1, 1].set_xlabel('Sex')
axes[1, 1].set_ylabel('Survival rate (%)')
axes[1, 1].set_ylim(0, 100)
for bar in axes[1, 1].patches:
    axes[1, 1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f'{bar.get_height():.0f}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

---

## Section 5 — Step 3: Preprocessing

Raw data is almost never ready for a model. We need to:

1. **Drop irrelevant columns** — `name`, `ticket`, `cabin` carry no useful signal for our model (or have too many missing values)
2. **Handle missing values** — `age` (177 missing): fill with the median; `embarked` (2 missing): fill with the mode
3. **Encode categoricals** — algorithms need numbers, not strings. `sex` → 0/1, `embarked` → one-hot dummies

In [ ]:
df = titanic.copy()

# --- Step 1: drop non-useful / high-missing columns ---
drop_cols = ['name', 'ticket', 'cabin', 'deck', 'embark_town', 'who', 'adult_male', 'alive', 'alone']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

print("After dropping columns:", list(df.columns))

# --- Step 2: missing values ---
print("\n=== Null counts BEFORE imputation ===")
print(df.isnull().sum())

# Fill age with median — robust to outliers
age_median = df['age'].median()
df['age'].fillna(age_median, inplace=True)

# Fill embarked with mode (most frequent port)
embarked_mode = df['embarked'].mode()[0]
df['embarked'].fillna(embarked_mode, inplace=True)

# Drop any remaining rows with nulls
df.dropna(inplace=True)

print("\n=== Null counts AFTER imputation ===")
print(df.isnull().sum())

# --- Step 3: encode categoricals ---
# sex: female=0, male=1
df['sex'] = df['sex'].map({'female': 0, 'male': 1})

# embarked: one-hot encode (drop_first avoids multicollinearity)
embarked_dummies = pd.get_dummies(df['embarked'], prefix='embarked', drop_first=True)
df = pd.concat([df.drop(columns='embarked'), embarked_dummies], axis=1)

# Convert bool columns to int
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("\nFinal columns:", list(df.columns))
print("Shape:", df.shape)
df.head()

---

## Section 6 — Step 4: Feature Engineering

Raw columns are a starting point — but we can often create **more informative** features by combining or transforming them.

**Intuition:** A person travelling alone might behave differently to a large family during an emergency. The original dataset has `sibsp` (siblings/spouses) and `parch` (parents/children) separately, but the *total family size* and whether someone is *alone* could be more predictive.

We'll create three new features:
- `family_size` — total people in the family group
- `is_alone` — binary flag: travelling alone?
- `age_group` — age discretised into 5 buckets (avoids the model over-fitting to exact ages)

In [ ]:
# Create new features
df['family_size'] = df['sibsp'] + df['parch'] + 1           # +1 for the passenger themselves
df['is_alone']    = (df['family_size'] == 1).astype(int)    # 1 if travelling alone
df['age_group']   = pd.cut(df['age'], bins=5, labels=False) # 0–4 buckets by equal-width

# How strongly does each new feature correlate with survival?
new_features = ['family_size', 'is_alone', 'age_group']
print("=== Correlation of new features with 'survived' ===")
print()
for feat in new_features:
    corr = df[feat].corr(df['survived'])
    direction = 'positive' if corr > 0 else 'negative'
    print(f"  {feat:<15}  r = {corr:+.3f}  ({direction} correlation)")

print()
print("family_size distribution:")
print(df['family_size'].value_counts().sort_index().to_string())
print(f"\nPassengers travelling alone: {df['is_alone'].sum()} / {len(df)} ({df['is_alone'].mean()*100:.0f}%)")

---

## Section 7 — Step 5: Train / Test Split

Before training anything, we set aside a **test set** — data the model will never see during training.

**Why keep the test set unseen?**  
If you evaluate on the same data you trained on, you're measuring *memorisation*, not *generalisation*. The test set simulates new, real-world data.

**Why `stratify=y`?**  
Our target class is imbalanced (~62% did not survive). Without stratify, a random split might accidentally put most survivors in one set, giving a misleading train/test comparison. `stratify=y` ensures both sets have the same class ratio as the full dataset.

In [ ]:
# Define features and target
FEATURES = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
            'embarked_Q', 'embarked_S',
            'family_size', 'is_alone', 'age_group']

# Keep only features that exist in df (get_dummies names can vary)
FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES]
y = df['survived']

# Split: 80% train, 20% test
# stratify=y → preserve class balance in both sets
# random_state=42 → reproducible split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=== Dataset split ===")
print(f"Total samples : {len(df)}")
print(f"Training set  : {len(X_train)} samples ({len(X_train)/len(df)*100:.0f}%)")
print(f"Test set      : {len(X_test)} samples ({len(X_test)/len(df)*100:.0f}%)")

print("\n=== Class balance (stratify check) ===")
print(f"Overall survival rate : {y.mean()*100:.1f}%")
print(f"Train survival rate   : {y_train.mean()*100:.1f}%")
print(f"Test survival rate    : {y_test.mean()*100:.1f}%")
print("\nAll three should be ~the same. That is what stratify=y guarantees.")

---

## Section 8 — Step 6: Feature Scaling

Many algorithms (including LogisticRegression) are sensitive to the scale of features. `age` ranges from 0–80 while `fare` ranges from 0–512 — a model might incorrectly treat `fare` as more important just because its numbers are larger.

`StandardScaler` transforms each feature to have **mean = 0** and **standard deviation = 1**:

$$z = \frac{x - \mu}{\sigma}$$

**Critical rule: fit the scaler on training data only, then transform both.**  
If you fit on all data (including test), you leak information about the test set into the training process — this is called **data leakage** and will make your results unrealistically optimistic.

In [ ]:
# fit_transform on TRAIN → learns mean and std from training data only
# transform on TEST  → applies the same learned parameters (no leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform train
X_test_scaled  = scaler.transform(X_test)        # transform test (no fitting)

# Show 'fare' before and after scaling
fare_idx = list(X_train.columns).index('fare')
fare_before = X_train['fare'].values
fare_after  = X_train_scaled[:, fare_idx]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fare_before, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title(f'fare — BEFORE scaling\nmean={fare_before.mean():.1f}, std={fare_before.std():.1f}')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')

axes[1].hist(fare_after, bins=30, color='#e67e22', edgecolor='white')
axes[1].set_title(f'fare — AFTER scaling\nmean={fare_after.mean():.2f}, std={fare_after.std():.2f}')
axes[1].set_xlabel('Value (z-score)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print("Before: fare has mean ≈", round(fare_before.mean(), 1), "and std ≈", round(fare_before.std(), 1))
print("After : fare has mean ≈", round(fare_after.mean(), 2),  "and std ≈", round(fare_after.std(), 2))

---

## Section 9 — Step 7: First Model — Logistic Regression

Logistic Regression is a strong baseline for binary classification. Despite the name it is a *classifier*, not a regressor. It models the probability that a passenger survived:

$$P(\text{survived}=1) = \frac{1}{1 + e^{-(w_0 + w_1 x_1 + w_2 x_2 + \ldots)}}$$

We start here because it is:
- Fast to train
- Interpretable (coefficients tell you feature importance direction)
- A good baseline to beat

After fitting, we evaluate with three tools:
- **Accuracy** — fraction of correct predictions (simple, but can mislead on imbalanced data)
- **Confusion matrix** — shows true positives, false positives, etc.
- **Classification report** — precision, recall, and F1 per class

In [ ]:
# Fit LogisticRegression
# max_iter=1000: maximum optimisation iterations (increase if convergence warning appears)
# random_state=42: reproducible results
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = lr.predict(X_train_scaled)
y_test_pred  = lr.predict(X_test_scaled)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc  = accuracy_score(y_test, y_test_pred)

print("=== Logistic Regression ===")
print(f"Training accuracy : {train_acc:.4f}  ({train_acc*100:.1f}%)")
print(f"Test accuracy     : {test_acc:.4f}  ({test_acc*100:.1f}%)")
print()

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_test_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted: 0', 'Predicted: 1'],
            yticklabels=['Actual: 0', 'Actual: 1'])
axes[0].set_title('Confusion Matrix (Test Set)', fontsize=13)
axes[0].set_ylabel('True label')
axes[0].set_xlabel('Predicted label')

# Normalised version
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1],
            xticklabels=['Predicted: 0', 'Predicted: 1'],
            yticklabels=['Actual: 0', 'Actual: 1'])
axes[1].set_title('Confusion Matrix — Normalised', fontsize=13)
axes[1].set_ylabel('True label')
axes[1].set_xlabel('Predicted label')

plt.tight_layout()
plt.show()

print("=== Classification Report ===")
print(classification_report(y_test, y_test_pred, target_names=['Did not survive', 'Survived']))

---

## Section 10 — The Bias-Variance Tradeoff

Every model makes two types of error:

| Error type | Also called | Cause | Symptom |
|------------|------------|-------|--------|
| **Bias** | Underfitting | Model too simple | High train error AND high test error |
| **Variance** | Overfitting | Model too complex | Low train error but HIGH test error |

**Plain-language examples:**
- A student who didn't study at all → high bias (bad on both practice problems and the real exam)
- A student who memorised every past exam verbatim → high variance (perfect on practice, fails on new questions)
- A student who understood the concepts → good balance (generalises to unseen questions)

We'll visualise this with polynomial regression on a sine curve.

In [ ]:
np.random.seed(42)

# Generate noisy sine data
N_TRAIN = 30
N_TEST  = 200

X_bv_train = np.sort(np.random.uniform(0, 1, N_TRAIN))
y_bv_train = np.sin(2 * np.pi * X_bv_train) + np.random.normal(0, 0.2, N_TRAIN)

X_bv_test = np.linspace(0, 1, N_TEST)
y_bv_test = np.sin(2 * np.pi * X_bv_test)  # true curve (no noise)

degrees = [1, 4, 15]
labels_bv = ['Degree 1\n(High Bias — Underfitting)',
             'Degree 4\n(Good Balance)',
             'Degree 15\n(High Variance — Overfitting)']
colors_bv = ['#e74c3c', '#2ecc71', '#9b59b6']

train_mses, test_mses = [], []

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Bias-Variance Tradeoff — Polynomial Fits on a Sine Curve', fontsize=14)

for ax, deg, label, color in zip(axes, degrees, labels_bv, colors_bv):
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    model.fit(X_bv_train.reshape(-1, 1), y_bv_train)

    y_pred_train = model.predict(X_bv_train.reshape(-1, 1))
    y_pred_test  = model.predict(X_bv_test.reshape(-1, 1))

    train_mse = mean_squared_error(y_bv_train, y_pred_train)
    test_mse  = mean_squared_error(y_bv_test, y_pred_test)
    train_mses.append(train_mse)
    test_mses.append(test_mse)

    ax.scatter(X_bv_train, y_bv_train, s=25, color='gray', alpha=0.7, label='Training data', zorder=3)
    ax.plot(X_bv_test, y_bv_test, 'k--', lw=1.5, label='True function', alpha=0.5)
    ax.plot(X_bv_test, y_pred_test, color=color, lw=2, label='Model fit')
    ax.set_title(f'{label}\nTrain MSE={train_mse:.3f}  Test MSE={test_mse:.3f}', fontsize=10)
    ax.set_ylim(-2.5, 2.5)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Plot train vs test MSE across a wider range of degrees
all_degrees = list(range(1, 16))
tr_mses, te_mses = [], []

for deg in all_degrees:
    m = Pipeline([('poly', PolynomialFeatures(degree=deg, include_bias=False)), ('reg', LinearRegression())])
    m.fit(X_bv_train.reshape(-1, 1), y_bv_train)
    tr_mses.append(mean_squared_error(y_bv_train, m.predict(X_bv_train.reshape(-1, 1))))
    te_mses.append(mean_squared_error(y_bv_test,  m.predict(X_bv_test.reshape(-1, 1))))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(all_degrees, tr_mses, 'o-', color='steelblue', label='Train MSE', lw=2)
ax.plot(all_degrees, te_mses, 's-', color='#e74c3c',  label='Test MSE',  lw=2)
ax.axvline(x=4, color='#2ecc71', linestyle='--', alpha=0.7, label='Sweet spot (~degree 4)')
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('MSE')
ax.set_title('Train vs Test MSE as model complexity grows')
ax.set_ylim(0, min(max(te_mses) * 1.1, 5))
ax.legend()
plt.tight_layout()
plt.show()

print("Key observation:")
print("  Train MSE always decreases as degree increases (model memorises more).")
print("  Test MSE decreases then increases — the U-shape is the bias-variance tradeoff.")
print("  The minimum of the test curve is the sweet spot.")

---

## Section 11 — Learning Curves

A **learning curve** plots model performance as a function of **training set size**. It answers: "Would more data help?"

| Shape | Diagnosis | Fix |
|-------|-----------|-----|
| Both curves plateau at a **low score** | High bias (underfitting) | More complex model, more features |
| Large **gap** between train and CV curves | High variance (overfitting) | More data, regularisation, simpler model |
| Curves **converge** at a high score | Well-fitted model | You're done! |

`learning_curve` in sklearn does cross-validation for us — it averages over multiple folds at each training size, giving a smoother and more reliable picture.

In [ ]:
# Use learning_curve on the full preprocessed X, y (it does its own splits internally)
# cv=5: 5-fold cross-validation at each training size
# train_sizes: evaluate at 10%, 20%, ..., 100% of training data
# scoring='accuracy': metric to report

lc_model = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(max_iter=1000, random_state=42))
])

train_sizes, train_scores, cv_scores = learning_curve(
    estimator   = lc_model,
    X           = X,
    y           = y,
    cv          = 5,
    train_sizes = np.linspace(0.1, 1.0, 10),
    scoring     = 'accuracy',
    n_jobs      = -1
)

# Compute mean and std across folds
train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
cv_mean    = cv_scores.mean(axis=1)
cv_std     = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(train_sizes, train_mean, 'o-', color='steelblue', label='Training score', lw=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color='steelblue')

ax.plot(train_sizes, cv_mean, 's-', color='#e74c3c', label='Cross-validation score', lw=2)
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='#e74c3c')

ax.set_title('Learning Curves — Logistic Regression on Titanic', fontsize=13)
ax.set_xlabel('Training set size')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.5, 1.0)
ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

gap = train_mean[-1] - cv_mean[-1]
print(f"At full training size:")
print(f"  Train accuracy  : {train_mean[-1]:.4f}")
print(f"  CV accuracy     : {cv_mean[-1]:.4f}")
print(f"  Gap (train - CV): {gap:.4f}")
print()
if gap > 0.1:
    print("Interpretation: moderate gap → some overfitting. Try regularisation or more data.")
elif cv_mean[-1] < 0.75:
    print("Interpretation: both curves plateau low → possible underfitting. Try richer features.")
else:
    print("Interpretation: curves converge at a decent score → reasonably well-fitted model.")

---

## Section 12 — Comparing Two Models

Real ML projects rarely stop at the first model. We compare several algorithms and pick the one that:
1. Has the best **test performance**
2. Has a small **train–test gap** (doesn't overfit)
3. Is appropriate for the use-case (speed, interpretability, etc.)

Here we compare:
- **Logistic Regression** — linear decision boundary, regularised, interpretable
- **Decision Tree (max_depth=5)** — non-linear, tree-based, captures interactions

`max_depth=5` limits the tree to 5 levels — without it, the tree would grow until it memorises every training example (pure overfitting).

In [ ]:
# Model definitions
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
}

results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    tr_acc  = accuracy_score(y_train, model.predict(X_train_scaled))
    tst_acc = accuracy_score(y_test,  model.predict(X_test_scaled))
    results.append({'Model': name, 'Train accuracy': tr_acc, 'Test accuracy': tst_acc,
                    'Gap': tr_acc - tst_acc})

results_df = pd.DataFrame(results).set_index('Model')
print("=== Model Comparison ===")
print(results_df.round(4).to_string())

# Bar chart comparison
fig, ax = plt.subplots(figsize=(9, 5))

x = np.arange(len(results_df))
width = 0.35

bars_train = ax.bar(x - width/2, results_df['Train accuracy'], width,
                    label='Train', color='steelblue', edgecolor='white')
bars_test  = ax.bar(x + width/2, results_df['Test accuracy'],  width,
                    label='Test',  color='#e74c3c', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(results_df.index, fontsize=11)
ax.set_ylabel('Accuracy')
ax.set_ylim(0.5, 1.0)
ax.set_title('Model Comparison: Train vs Test Accuracy', fontsize=13)
ax.legend()

for bar in bars_train:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars_test:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\nModel selection intuition:")
print("  - A large Train-Test gap signals overfitting.")
print("  - Prefer the model with the best TEST score AND smallest gap.")
print("  - If both models score similarly, prefer the simpler/more interpretable one.")

---

## Section 13 — Practice Exercises

Complete the three exercises below. Each has a starter cell and a solution cell (collapsed). Try on your own before looking at the solution.

---

### Exercise 1 — Overfitting in action

Fit a `DecisionTreeClassifier` with `max_depth=1` and another with `max_depth=20`. Print the train and test accuracy for each. What do you notice?

**Expected insight:** `max_depth=1` underfits (high bias); `max_depth=20` overfits (high variance, tiny test accuracy drop).

In [ ]:
# --- Exercise 1: YOUR CODE HERE ---

# Hint: loop over depths = [1, 20]
# For each depth:
#   1. Create DecisionTreeClassifier(max_depth=depth, random_state=42)
#   2. Fit on X_train_scaled, y_train
#   3. Print train accuracy and test accuracy

In [ ]:
# --- Exercise 1 SOLUTION ---

for depth in [1, 20]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_train_scaled, y_train)
    tr  = accuracy_score(y_train, dt.predict(X_train_scaled))
    tst = accuracy_score(y_test,  dt.predict(X_test_scaled))
    print(f"max_depth={depth:>2}  |  Train: {tr:.4f}  |  Test: {tst:.4f}  |  Gap: {tr-tst:.4f}")

print()
print("max_depth=1  → underfitting: even training accuracy is low (model too simple)")
print("max_depth=20 → overfitting : train accuracy near 1.0 but test accuracy drops")

---

### Exercise 2 — Extract a `title` feature from the Name column

The original Titanic `name` column contains titles like `Mr`, `Mrs`, `Miss`, `Master`, `Dr`. These carry social status and survival signal (children labelled `Master` had higher survival rates).

**Task:**
1. Reload the raw titanic dataset
2. Extract the title using a regex on the `name` column
3. Print the value counts of the extracted titles
4. Print the survival rate for each title

In [ ]:
# --- Exercise 2: YOUR CODE HERE ---

raw = sns.load_dataset('titanic')

# Hint: use raw['name'].str.extract(r', ([A-Za-z]+)\.')
# The name format is: 'Braund, Mr. Owen Harris'
# The title is between ', ' and '.'

# Your code:


In [ ]:
# --- Exercise 2 SOLUTION ---

raw = sns.load_dataset('titanic')

# Extract title using regex
raw['title'] = raw['name'].str.extract(r', ([A-Za-z]+)\.')

print("=== Title value counts ===")
print(raw['title'].value_counts())

print("\n=== Survival rate by title ===")
survival_by_title = raw.groupby('title')['survived'].agg(['mean', 'count'])
survival_by_title.columns = ['survival_rate', 'count']
survival_by_title = survival_by_title[survival_by_title['count'] >= 3]
print(survival_by_title.sort_values('survival_rate', ascending=False).round(3).to_string())

print("\nInsight: 'Mrs' and 'Miss' have very high survival rates (women first policy).")
print("'Master' (young boys) also survived more than 'Mr' (adult men).")

---

### Exercise 3 — Load Iris and fit a model

Switch to the famous **Iris** dataset — 150 flower samples across 3 species, with 4 numeric features (sepal/petal length and width). No missing values, no encoding needed.

**Task:**
1. Load with `load_iris()` and create a DataFrame
2. Split into train/test (80/20, stratified)
3. Scale the features
4. Fit a `LogisticRegression` and print train + test accuracy
5. Print the classification report

In [ ]:
# --- Exercise 3: YOUR CODE HERE ---

# Hint:
# iris = load_iris()
# X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
# y_iris = iris.target

# Your code:


In [ ]:
# --- Exercise 3 SOLUTION ---

# Load
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
y_iris = iris.target

print("Iris dataset shape:", X_iris.shape)
print("Classes:", list(iris.target_names))
print("Class distribution:", pd.Series(y_iris).value_counts().to_dict())

# Split
Xi_train, Xi_test, yi_train, yi_test = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

# Scale
sc = StandardScaler()
Xi_train_s = sc.fit_transform(Xi_train)
Xi_test_s  = sc.transform(Xi_test)

# Fit
lr_iris = LogisticRegression(max_iter=1000, random_state=42)
lr_iris.fit(Xi_train_s, yi_train)

tr_acc  = accuracy_score(yi_train, lr_iris.predict(Xi_train_s))
tst_acc = accuracy_score(yi_test,  lr_iris.predict(Xi_test_s))

print(f"\nTrain accuracy : {tr_acc:.4f}")
print(f"Test accuracy  : {tst_acc:.4f}")

print("\n=== Classification Report ===")
print(classification_report(yi_test, lr_iris.predict(Xi_test_s), target_names=iris.target_names))

print("Iris is a 'clean' dataset — good scores expected. Real data is messier (like Titanic).")

---

## Summary

In this tutorial you covered the full supervised ML workflow:

| Step | What we did |
|------|-------------|
| 1. What is ML | 3 types, real-world examples, KNN by hand |
| 2. Regression vs Classification | Synthetic datasets, visual comparison |
| 3. ML Workflow | 7-step checklist |
| 4. Data & EDA | Loaded Titanic, null audit, 4-panel figure |
| 5. Preprocessing | Drop columns, fill nulls, encode categoricals |
| 6. Feature Engineering | family_size, is_alone, age_group |
| 7. Train/Test Split | 80/20, stratify explained |
| 8. Feature Scaling | StandardScaler, data leakage warning |
| 9. First Model | LogisticRegression, confusion matrix, classification report |
| 10. Bias-Variance | Polynomial fits, train vs test MSE curve |
| 11. Learning Curves | Diagnosing underfitting vs overfitting |
| 12. Model Comparison | LogisticRegression vs DecisionTree, bar chart |
| 13. Exercises | Overfitting, title feature, Iris end-to-end |

### Next steps

- **5.2** — Supervised Learning: Regression in depth (Linear, Ridge, Lasso)
- **5.3** — Supervised Learning: Classification in depth (trees, ensembles, SVM)
- **5.4** — Model evaluation: cross-validation, ROC-AUC, hyperparameter tuning